
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Creating Bronze Tables from JSON Files
### Duration: ~ 15 minutes

In this lab you will ingest a JSON file as UC table and then flatten the JSON formatted string column.

### Learning Objectives
  - Inspect a raw JSON file.
  - Read in JSON files to a UC table and flatten the JSON formatted string column.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.3/images/20261001T131157Z/Data Ingestion with LakeFlow Connect/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>



## A. Classroom Setup

Run the following cell to configure your working environment for this notebook.

In [0]:
%run ./Includes/Classroom-Setup-13L

Run the cell below to view your default catalog and schema. Notice that your default catalog is **labuser123_678** and your default schema is your unique **data_ingestion** schema.

**NOTE:** The default catalog and schema are pre-configured for you to avoid the need to specify the three-level name when writing your tables (i.e., catalog.schema.table).

In [0]:
SELECT current_catalog(), current_schema()

## B. Lab - JSON Ingestion
**Scenario:** You are working with your data team on ingesting a JSON file into Databricks. Your job is to ingest the JSON file as is into a bronze table, then create a second bronze table that flattens the JSON formatted string column in the raw bronze table for downstream processing.

### B1. Inspect the Dataset

1. In the cell below, view the value of the variable `my_catalog`. This variable is created within the classroom setup script to dynamically reference your unique volume. You can concatenate the `my_catalog` SQL variable with a string to specify a specific subdirectory in your specific volume.

   Run the cell to view the data in the `/Volumes/your-labuser-name/data_ingestion/landing_folder/json_demo_files/lab_kafka_events.json` file in the location from above.

**Note:** Instead of using the `my_catalog` variable, you could also specify the path name directly by right clicking on your volume and selecting **Copy volume path**.

In [0]:
%python
spark.sql(f'''
          SELECT * 
          FROM json.`/Volumes/{my_catalog}/data_ingestion/landing_folder/json_demo_files/lab_kafka_events.json`
          ''').display()

### B2. Create the Raw Bronze Table

Inspect and run the code below to ingest the raw JSON file  `/Volumes/your-labuser-name/data_ingestion/landing_folder/json_demo_files/lab_kafka_events.json` and create the **lab13_lab_kafka_events_raw** table.

Notice the following:
- The **value** column is decoded.
- The **decoded_value** column was created and returns the decoded column as a JSON-formatted string.


In [0]:
CREATE OR REPLACE TABLE lab13_lab_kafka_events_raw
AS
SELECT 
  *,
  cast(unbase64(value) as STRING) as decoded_value
FROM read_files(
        '/Volumes/' || my_catalog || '/data_ingestion/landing_folder/json_demo_files/lab_kafka_events.json',
        format => "json", 
        schema => '''
          key STRING, 
          timestamp DOUBLE, 
          value STRING
        ''',
        rescueddatacolumn => '_rescued_data'
      );

-- View the table
SELECT *
FROM lab13_lab_kafka_events_raw;

### B3. Create the Flattened Bronze Table

1. Your goal is to flatten the JSON formatted string column **decoded_value** from the table **lab13_lab_kafka_events_raw** to create a new table named **lab13_lab_kafka_events_flattened** for downstream processing. The table should contain the following columns:
    - **key**
    - **timestamp**
    - **user_id**
    - **event_type**
    - **event_timestamp**
    - **items**

    You can use whichever technique you prefer:

    - Parse the JSON formatted string (easiest) to flatten
      - [Query JSON strings](https://docs.databricks.com/aws/en/semi-structured/json):

    - Convert the JSON formatted string as a VARIANT and flatten
      - [parse_json function](https://docs.databricks.com/gcp/en/sql/language-manual/functions/parse_json)

    - Convert the JSON formatted string to a STRUCT and flatten
      - [schema_of_json function](https://docs.databricks.com/aws/en/sql/language-manual/functions/schema_of_json)
      - [from_json function](https://docs.databricks.com/gcp/en/sql/language-manual/functions/from_json)

**NOTE:** View the lab solution notebook to view the solutions for each.

2. To begin, run the code below to view the final solution table **lab13_lab_kafka_events_flattened_solution**. This will give you an idea of what your final table should look like.

  **NOTE**: Depending on your solution, the data types of the columns may vary slightly.  


##### Optional Challenge

  As a challenge, after flattening the table, try converting the data types accordingly. Depending on your skill set, you may not convert all columns to the correct data types within the allotted time.

  - **key** STRING
  - **timestamp** DOUBLE
  - **user_id** STRING
  - **event_type** STRING
  - **event_timestamp** TIMESTAMP
  - **items** (STRUCT or VARIANT) depending on the method you used.

In [0]:
SELECT *
FROM lab13_lab_kafka_events_flattened_solution

3. Write the query in the cell below to read the **lab_kafka_events_raw** table and create the flattened table **lab13_lab_kafka_events_flattened** following the requirements from above.

In [0]:
<FILL-IN>

<details>
<summary><strong>Click to reveal answer</strong></summary>

<!-- Hidden source for the Python answer -->
<textarea id="raw-answer-python-customer-id-analysis" style="display:none;">
---- Parse the JSON formatted STRING
CREATE OR REPLACE TABLE lab13_lab_kafka_events_flattened_str
AS
SELECT 
  key,
  timestamp,
  decoded_value:user_id,
  decoded_value:event_type,
  cast(decoded_value:event_timestamp AS TIMESTAMP),
  from_json(decoded_value:items,'ARRAY<STRUCT<item_id: STRING, price_usd: DOUBLE, quantity: BIGINT>>') AS items
FROM lab13_lab_kafka_events_raw;


---- Display the table
SELECT *
FROM lab13_lab_kafka_events_flattened_str;

</textarea>

<div class="code-block-dark" data-language="python" data-source="raw-answer-python-customer-id-analysis"></div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism-okaidia.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block-dark').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');

        var lang = block.getAttribute('data-language') || 'python';
        var sourceId = block.getAttribute('data-source');

        // Use textarea as the single source of truth so multiline f-strings render correctly [web:137][web:143].
        var raw;
        if (sourceId) {
            var rawEl = document.getElementById(sourceId);
            if (!rawEl) return;
            raw = rawEl.value;
        } else {
            raw = block.textContent;
        }

        var code = raw.trim(); // preserve original trimming behavior [web:4]

        var id = 'code-dark-' + Math.random().toString(36).substr(2, 9);

        block.innerHTML =
            '<div style="position:relative;margin:16px 0;max-width:100%;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#555;color:#fff;border:1px solid #666;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#272822;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #444;max-width:100%;box-sizing:border-box;">' +
                    '<code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;word-wrap:break-word;white-space:pre-wrap;"></code>' +
                '</pre>' +
            '</div>';

        var codeEl = document.getElementById(id);
        codeEl.textContent = code; // Prism highlights content inside <pre><code> with language-* class [web:114][web:75]
        Prism.highlightElement(codeEl);

        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

In [0]:
<FILL-IN>

<details>
<summary><strong>Click to reveal answer</strong></summary>

<!-- Hidden source for the Python answer -->
<textarea id="raw-answer-python-customer-id-analysis" style="display:none;">
---- Convert the JSON formatted string as a VARIANT
---- NOTE: The VARIANT decoded_value_variant column is included in this solution to display the column
---- NOTE: Variant data type will not work on Serverless Version 1.
CREATE OR REPLACE TABLE lab13_lab_kafka_events_flattened_variant
AS
SELECT
  key,
  timestamp,
  parse_json(decoded_value) AS decoded_value_variant,
  cast(decoded_value_variant:user_id AS STRING),
  decoded_value_variant:event_type :: STRING,
  decoded_value_variant:event_timestamp :: TIMESTAMP,
  decoded_value_variant:items
FROM lab13_lab_kafka_events_raw;


---- Display the table
SELECT *
FROM lab13_lab_kafka_events_flattened_variant;
</textarea>

<div class="code-block-dark" data-language="python" data-source="raw-answer-python-customer-id-analysis"></div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism-okaidia.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block-dark').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');

        var lang = block.getAttribute('data-language') || 'python';
        var sourceId = block.getAttribute('data-source');

        // Use textarea as the single source of truth so multiline f-strings render correctly [web:137][web:143].
        var raw;
        if (sourceId) {
            var rawEl = document.getElementById(sourceId);
            if (!rawEl) return;
            raw = rawEl.value;
        } else {
            raw = block.textContent;
        }

        var code = raw.trim(); // preserve original trimming behavior [web:4]

        var id = 'code-dark-' + Math.random().toString(36).substr(2, 9);

        block.innerHTML =
            '<div style="position:relative;margin:16px 0;max-width:100%;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#555;color:#fff;border:1px solid #666;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#272822;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #444;max-width:100%;box-sizing:border-box;">' +
                    '<code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;word-wrap:break-word;white-space:pre-wrap;"></code>' +
                '</pre>' +
            '</div>';

        var codeEl = document.getElementById(id);
        codeEl.textContent = code; // Prism highlights content inside <pre><code> with language-* class [web:114][web:75]
        Prism.highlightElement(codeEl);

        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

In [0]:
<FILL-IN>

<details>
<summary><strong>Click to reveal answer</strong></summary>

<!-- Hidden source for the Python answer -->
<textarea id="raw-answer-python-customer-id-analysis" style="display:none;">
---- Convert the JSON formatted string as a STRUCT
---- Return the structure of the JSON formatted string
SELECT schema_of_json(decoded_value)
FROM lab13_lab_kafka_events_raw
LIMIT 1;


---- Use the JSON structure from above within the from_json function to convert the JSON formatted string to a STRUCT
---- NOTE: The STRUCT decoded_value_struct column is included in this solution to display the column
CREATE OR REPLACE TABLE lab13_lab_kafka_events_flattened_struct
AS
SELECT
  key,
  timestamp,
  from_json(decoded_value, 'STRUCT<event_timestamp: STRING, event_type: STRING, items: ARRAY<STRUCT<item_id: STRING, price_usd: DOUBLE, quantity: BIGINT>>, user_id: STRING>') AS decoded_value_struct,
  decoded_value_struct.user_id,
  decoded_value_struct.event_type,
  cast(decoded_value_struct.event_timestamp AS TIMESTAMP),
  decoded_value_struct.items
FROM lab13_lab_kafka_events_raw;


---- Display the table
SELECT *
FROM lab13_lab_kafka_events_flattened_struct;
</textarea>

<div class="code-block-dark" data-language="python" data-source="raw-answer-python-customer-id-analysis"></div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism-okaidia.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block-dark').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');

        var lang = block.getAttribute('data-language') || 'python';
        var sourceId = block.getAttribute('data-source');

        // Use textarea as the single source of truth so multiline f-strings render correctly [web:137][web:143].
        var raw;
        if (sourceId) {
            var rawEl = document.getElementById(sourceId);
            if (!rawEl) return;
            raw = rawEl.value;
        } else {
            raw = block.textContent;
        }

        var code = raw.trim(); // preserve original trimming behavior [web:4]

        var id = 'code-dark-' + Math.random().toString(36).substr(2, 9);

        block.innerHTML =
            '<div style="position:relative;margin:16px 0;max-width:100%;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#555;color:#fff;border:1px solid #666;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#272822;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #444;max-width:100%;box-sizing:border-box;">' +
                    '<code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;word-wrap:break-word;white-space:pre-wrap;"></code>' +
                '</pre>' +
            '</div>';

        var codeEl = document.getElementById(id);
        codeEl.textContent = code; // Prism highlights content inside <pre><code> with language-* class [web:114][web:75]
        Prism.highlightElement(codeEl);

        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>